#  IshVenom — Vision Classifier Training

**EfficientNet-Lite0** fine-tuned on 20 African snake species + unknown class.

### Kaggle Settings
| Setting | Value |
|---------|-------|
| Accelerator | GPU T4 × 2 |
| Internet | ON |
| Attached Dataset | `kwakyeishmael/snakes-africa` |

### Outputs (saved to `/kaggle/working/`)
- `venomwise-vision.tflite` — INT8 quantized model for Android
- `labels.json` — class index → species name mapping
- `best_vision.pt` — PyTorch checkpoint for future fine-tuning

**Runtime:** ~2 hours on T4 × 2

## 1 · Install Dependencies
Install `timm` (PyTorch Image Models), `albumentations` (augmentations),
and the ONNX→TF→TFLite export chain.

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm>=1.0.0",
    "albumentations>=1.4.0",
    "onnx>=1.16.0",
    "onnxruntime>=1.19.0",
    "onnx-tf>=1.10.0",
    "tensorflow>=2.16.0",
])
print(" Dependencies installed")

## 2 · Configuration
All hyperparameters and paths in one place.
The dataset is attached at `/kaggle/input/datasets/kwakyeishmael/snakes-africa/`.

In [ ]:
import os, json, random
from pathlib import Path

# ── Paths ──
# IMPORTANT: Kaggle mounts datasets at /kaggle/input/datasets/<owner>/<name>/
# If the path does not exist, check: !ls /kaggle/input/
KAGGLE_ROOT  = Path("/kaggle/input/datasets/kwakyeishmael/snakes-africa")
SPLITS_DIR   = KAGGLE_ROOT / "data/processed/splits"
WORKING_DIR  = Path("/kaggle/working")

# ── Hyperparameters ──
NUM_CLASSES  = 21          # 20 priority species + unknown
EPOCHS       = 25
BATCH_SIZE   = 64          # EfficientNet-Lite0 is small; 64 fits on T4
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1
IMG_SIZE     = 224
SEED         = 42
NUM_WORKERS  = 2

random.seed(SEED)

# ── Verify paths exist before we proceed ──
assert KAGGLE_ROOT.exists(), (
    f"Dataset not found at {KAGGLE_ROOT}. "
    f"Check: !ls /kaggle/input/ and update KAGGLE_ROOT"
)
assert SPLITS_DIR.exists(), f"Splits not found at {SPLITS_DIR}"

split_files = list(SPLITS_DIR.glob("*.jsonl"))
print(f"KAGGLE_ROOT: {KAGGLE_ROOT}")
print(f"Splits found: {[f.name for f in split_files]}")
print(f"Config: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}")


## 3 · Build Label Map
Read all species names from the JSONL split files and create a
sorted `species → index` mapping. The `unknown` class is always included.

In [ ]:
def build_label_map(splits_dir: Path) -> dict[str, int]:
    """Scan all split JSONL files and build a sorted label map."""
    species: set[str] = set()
    for split in ("train", "val", "test"):
        p = splits_dir / f"{split}.jsonl"
        if p.exists():
            with p.open() as f:
                for line in f:
                    row = json.loads(line)
                    species.add(row["species"])
    species.add("unknown")
    return {s: i for i, s in enumerate(sorted(species))}

LABEL_MAP = build_label_map(SPLITS_DIR)
print(f" {len(LABEL_MAP)} classes:")
for name, idx in LABEL_MAP.items():
    print(f"  {idx:2d} → {name}")

## 4 · Data Augmentation Transforms
**Training:** aggressive augmentations (random crop, flip, color jitter,
blur, coarse dropout) to prevent overfitting on a small dataset.

**Validation/Test:** deterministic resize + center crop only.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

TRAIN_TRANSFORM = A.Compose([
    A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.6, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

VAL_TRANSFORM = A.Compose([
    A.Resize(256, 256),
    A.CenterCrop(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print(" Transforms defined")

## 5 · Dataset Class
Reads rows from the JSONL split files. Each row has a `path` (relative to
the zip root) and a `species` name. Images are loaded with PIL and
augmented with the transforms above.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np


class SnakeDataset(Dataset):
    """PyTorch Dataset that reads from IshVenom JSONL split files."""

    def __init__(self, jsonl_path: Path, label_map: dict[str, int], transform: A.Compose):
        self.rows: list[dict] = []
        with jsonl_path.open() as f:
            for line in f:
                self.rows.append(json.loads(line))
        self.label_map = label_map
        self.transform = transform

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int):
        row = self.rows[idx]
        # Paths in JSONL are relative (e.g. data/raw/inat/...) — resolve from Kaggle root
        img_path = KAGGLE_ROOT / row["path"]
        img = np.array(Image.open(img_path).convert("RGB"))
        img = self.transform(image=img)["image"]
        label = self.label_map.get(row["species"], self.label_map["unknown"])
        return img, label


print(" SnakeDataset class defined")

## 6 · Create DataLoaders
Load the train/val/test splits and create PyTorch DataLoaders with
batching, shuffling (train only), and pinned memory for GPU transfer.

In [ ]:
train_ds = SnakeDataset(SPLITS_DIR / "train.jsonl", LABEL_MAP, TRAIN_TRANSFORM)
val_ds   = SnakeDataset(SPLITS_DIR / "val.jsonl",   LABEL_MAP, VAL_TRANSFORM)
test_ds  = SnakeDataset(SPLITS_DIR / "test.jsonl",  LABEL_MAP, VAL_TRANSFORM)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f" DataLoaders created")
print(f"   Train: {len(train_ds):,} images | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

## 7 · Initialize Model
**EfficientNet-Lite0** from `timm` — a mobile-optimized architecture.
Pretrained on ImageNet, we replace the classifier head with our 21-class
output. Uses DataParallel if multiple GPUs are available.

In [ ]:
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}", end="")
if torch.cuda.is_available():
    print(f" ({torch.cuda.get_device_name(0)})")
    if torch.cuda.device_count() > 1:
        print(f"  → {torch.cuda.device_count()} GPUs detected")
else:
    print()

model = timm.create_model("efficientnet_lite0", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

# Multi-GPU if available
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
    print(f" DataParallel enabled across {torch.cuda.device_count()} GPUs")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

total_params = sum(p.numel() for p in model.parameters())
print(f" Model initialized: {total_params:,} parameters")

## 8 · Training Loop
Train for `EPOCHS` epochs. After each epoch, evaluate on the validation set.
Save the best checkpoint by **top-3 accuracy** (our primary metric, target ≥ 85%).

In [ ]:
def accuracy_topk(outputs: torch.Tensor, labels: torch.Tensor, k: int = 3) -> float:
    """Compute top-k accuracy."""
    _, pred = outputs.topk(k, dim=1)
    correct = pred.eq(labels.view(-1, 1).expand_as(pred)).sum().item()
    return correct / labels.size(0)


best_val_top3 = 0.0
CKPT_PATH = WORKING_DIR / "best_vision.pt"

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    running_loss = total = correct1 = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, pred = out.max(1)
        correct1 += pred.eq(labels).sum().item()
        total += labels.size(0)
    train_loss = running_loss / len(train_loader)
    train_acc1 = 100 * correct1 / total

    # ── Validate ──
    model.eval()
    val_top1 = val_top3 = val_total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            val_top1 += accuracy_topk(out, labels, 1) * labels.size(0)
            val_top3 += accuracy_topk(out, labels, 3) * labels.size(0)
            val_total += labels.size(0)
    v1 = 100 * val_top1 / val_total
    v3 = 100 * val_top3 / val_total
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"loss={train_loss:.4f} | train_top1={train_acc1:.1f}% | "
        f"val_top1={v1:.1f}% | val_top3={v3:.1f}%"
    )

    if v3 > best_val_top3:
        best_val_top3 = v3
        raw_model = model.module if hasattr(model, "module") else model
        torch.save({"state_dict": raw_model.state_dict(), "label_map": LABEL_MAP}, CKPT_PATH)
        print(f"   Saved best checkpoint (val top-3: {v3:.1f}%)")

print(f"\n Best val top-3 accuracy: {best_val_top3:.1f}% (target ≥85%)")

## 9 · Export to ONNX
Convert the best PyTorch checkpoint to ONNX format.
This is the first step of the PyTorch → ONNX → TF → TFLite INT8 pipeline.

In [ ]:
import onnx

# Load best checkpoint
ckpt = torch.load(CKPT_PATH, map_location="cpu")
export_model = timm.create_model("efficientnet_lite0", pretrained=False, num_classes=NUM_CLASSES)
export_model.load_state_dict(ckpt["state_dict"])
export_model.eval()

# PyTorch → ONNX
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path = WORKING_DIR / "vision_model.onnx"
torch.onnx.export(
    export_model, dummy, str(onnx_path),
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch_size"}},
    opset_version=13,
)
print(f" ONNX exported: {onnx_path} ({onnx_path.stat().st_size / 1024:.1f} KB)")

## 10 · Convert ONNX → TensorFlow SavedModel
The `onnx-tf` library converts the ONNX graph to a TF SavedModel,
which is required as input for the TFLite converter.

In [ ]:
try:
    from onnx_tf.backend import prepare
    
    tf_dir = WORKING_DIR / "vision_model_tf"
    onnx_model = onnx.load(str(onnx_path))
    tf_rep = prepare(onnx_model)
    tf_rep.export_graph(str(tf_dir))
    print(f"TF SavedModel exported: {tf_dir}")
    USE_ONNX_TF = True
except Exception as e:
    print(f"onnx-tf failed: {e}")
    print("Falling back to float16 TFLite (no INT8 quantization)")
    USE_ONNX_TF = False


## 11 · Convert to TFLite INT8 (for Android)
Quantize to INT8 using **real validation images** as the representative
dataset for calibration. This is critical — random noise would destroy
the activation ranges and silently tank accuracy.

The output `.tflite` is what `react-native-fast-tflite` loads on the phone.

In [ ]:
import tensorflow as tf

if USE_ONNX_TF:
    # ── Full INT8 quantization (best, uses onnx-tf SavedModel) ──
    val_rows: list[dict] = []
    with (SPLITS_DIR / "val.jsonl").open() as f:
        for line in f:
            val_rows.append(json.loads(line))
    calib_rows = random.sample(val_rows, min(100, len(val_rows)))

    def representative_dataset():
        """Yield real validation images for INT8 calibration."""
        for row in calib_rows:
            img = np.array(Image.open(KAGGLE_ROOT / row["path"]).convert("RGB"), dtype=np.float32)
            img = A.Compose([
                A.Resize(256, 256),
                A.CenterCrop(IMG_SIZE, IMG_SIZE),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ])(image=img)["image"]
            yield [img[np.newaxis, ...]]  # TFLite expects NHWC

    converter = tf.lite.TFLiteConverter.from_saved_model(str(tf_dir))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type  = tf.int8
    converter.inference_output_type = tf.int8
    tflite_model = converter.convert()
    quant_type = "INT8"
else:
    # ── Fallback: ONNX Runtime → direct float16 TFLite ──
    # If onnx-tf failed, export directly from ONNX via ORT
    import onnxruntime as ort
    print("Using ONNX Runtime fallback for TFLite conversion...")
    # Export a simpler float16 TFLite via TF lite from ONNX
    # Actually, the simplest fallback: just save the .pt and labels.json
    # and convert to TFLite on your Mac later
    print("Saving PyTorch checkpoint + labels. Convert to TFLite locally:")
    print("  pip install onnx-tf tensorflow")
    print("  python -c 'from ml.train.vision_export import export; export()'")
    tflite_model = None
    quant_type = "NONE (fallback)"

if tflite_model:
    tflite_path = WORKING_DIR / "venomwise-vision.tflite"
    tflite_path.write_bytes(tflite_model)
    print(f"TFLite {quant_type}: {tflite_path} ({len(tflite_model) / 1024 / 1024:.2f} MB)")
else:
    print("TFLite export skipped — convert locally using the .pt checkpoint")


## 12 · Save Labels & Final Summary
Save the `labels.json` file (index → species name) that the mobile app
reads alongside the `.tflite` model. Print the final summary.

In [ ]:
labels_path = WORKING_DIR / "labels.json"
idx_to_species = {str(v): k for k, v in LABEL_MAP.items()}
labels_path.write_text(json.dumps(idx_to_species, indent=2, ensure_ascii=False))
print(f"labels.json saved: {labels_path}")

print("\n" + "=" * 60)
print("SUMMARY")
print(f"  Best val top-3 accuracy : {best_val_top3:.1f}%  (target >=85%)")
if tflite_model:
    print(f"  TFLite model size       : {len(tflite_model) / 1024 / 1024:.2f} MB")
else:
    print(f"  TFLite                  : not exported (convert .pt locally)")
print(f"  Checkpoint              : {CKPT_PATH}")
print("=" * 60)
print("\nDownload from /kaggle/working/:")
print("   1. venomwise-vision.tflite  -> apps/mobile/assets/models/")
print("   2. labels.json              -> apps/mobile/assets/models/")
print("   3. best_vision.pt           -> ml/runs/vision/ (keep for later)")


## 13 · Push to HuggingFace (Backup)
Upload the trained model files to HuggingFace so they are not lost
when the Kaggle session ends. Uses the same write token from the Gemma notebook.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface-hub"])

from huggingface_hub import HfApi

HF_USERNAME = "CalyxIsh"
REPO_ID = f"{HF_USERNAME}/ishvenom-vision-classifier"
HF_WRITE_TOKEN = "hf_REDACTED_USE_KAGGLE_SECRET_HF_TOKEN"

api = HfApi(token=HF_WRITE_TOKEN)

# Create the repo if it does not exist
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

# Upload TFLite model
if (WORKING_DIR / "venomwise-vision.tflite").exists():
    api.upload_file(
        path_or_fileobj=str(WORKING_DIR / "venomwise-vision.tflite"),
        path_in_repo="venomwise-vision.tflite",
        repo_id=REPO_ID,
        repo_type="model",
    )
    print("Uploaded venomwise-vision.tflite")

# Upload labels.json
if (WORKING_DIR / "labels.json").exists():
    api.upload_file(
        path_or_fileobj=str(WORKING_DIR / "labels.json"),
        path_in_repo="labels.json",
        repo_id=REPO_ID,
        repo_type="model",
    )
    print("Uploaded labels.json")

# Upload PyTorch checkpoint
if (WORKING_DIR / "best_vision.pt").exists():
    api.upload_file(
        path_or_fileobj=str(WORKING_DIR / "best_vision.pt"),
        path_in_repo="best_vision.pt",
        repo_id=REPO_ID,
        repo_type="model",
    )
    print("Uploaded best_vision.pt")

print(f"\nAll files pushed to: https://huggingface.co/{REPO_ID}")
